# 4.3 — Decision Trees: Every Concept Explained From Scratch
**Deep Theory + Visuals + From-Scratch Code — Nothing Skipped**

## Table of Contents
1. What is a Decision Tree — the core idea
2. Gini Impurity — how splits are scored
3. Information Gain & Entropy — the alternative criterion
4. How the tree finds the best split (exhaustive search)
5. Tree depth and leaves — anatomy of a tree
6. Overfitting — why unlimited trees always overfit
7. Pruning — max_depth, min_samples_leaf, min_samples_split
8. Feature importance — how trees rank features
9. Handling numerical vs categorical features
10. Decision Trees for Regression (CART)
11. Instability — why small data changes flip the tree
12. Full Decision Tree from scratch — complete implementation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.tree import plot_tree, export_text
from sklearn.datasets import make_classification, make_moons
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
np.random.seed(42)
print("Ready.")

---
## Concept 1 — What is a Decision Tree

A Decision Tree is a flowchart of yes/no questions. Each internal node asks a question about one feature. Each leaf gives a prediction.

### The core question: What makes a good split?
At each node, the tree asks: *which feature and which threshold best separates the classes?*

It tries **every feature** and **every possible threshold** on that feature.
It picks the combination that results in the **purest children** — measured by Gini or Entropy.

### CART — Classification and Regression Trees
sklearn's DecisionTreeClassifier uses CART:
- Binary splits only (always splits into exactly 2 children)
- Greedy — picks the locally best split at each node
- No look-ahead — doesn't consider what comes two levels below

### Anatomy
```
          [Root Node]          ← asks the best overall question
         /           \
   [Internal]     [Internal]   ← ask further questions
    /     \
[Leaf]  [Leaf]                 ← give final prediction (class or value)
```

In [ ]:
# === Concept 1: Visualise a simple trained tree ===

# Simple 2-feature dataset
np.random.seed(42)
X_demo = np.random.randn(200, 2)
y_demo = ((X_demo[:,0] > 0) & (X_demo[:,1] > 0)).astype(int)

dt_demo = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_demo.fit(X_demo, y_demo)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: decision boundary
xx, yy = np.meshgrid(np.linspace(-3,3,200), np.linspace(-3,3,200))
Z = dt_demo.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
axes[0].contourf(xx, yy, Z, alpha=0.25, cmap='RdBu')
axes[0].contour(xx, yy, Z, colors='gray', linewidths=0.8)
axes[0].scatter(X_demo[y_demo==0,0], X_demo[y_demo==0,1], c='steelblue', s=25, alpha=0.7, label='Class 0')
axes[0].scatter(X_demo[y_demo==1,0], X_demo[y_demo==1,1], c='coral',     s=25, alpha=0.7, label='Class 1')
axes[0].set_title('Decision Tree Boundary\n(Axis-aligned splits = rectangular regions)', fontsize=10)
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Right: tree structure
plot_tree(dt_demo, feature_names=['Feature 0','Feature 1'],
          class_names=['Class 0','Class 1'],
          filled=True, rounded=True, fontsize=9, ax=axes[1])
axes[1].set_title('Tree Structure — each node = one yes/no question', fontsize=10)

plt.tight_layout(); plt.show()

# Print text rules
print("=== Tree Rules (Text) ===")
print(export_text(dt_demo, feature_names=['Feature_0','Feature_1']))

---
## Concept 2 — Gini Impurity

Gini measures how **mixed** a node is.

$$Gini = 1 - \sum_{i=1}^{C} p_i^2$$

Where $p_i$ = proportion of class $i$ in the node.

| Node | Gini | Meaning |
|---|---|---|
| All one class | 0.0 | Perfectly pure ← best |
| 50/50 split | 0.5 | Maximally impure ← worst |
| 70/30 split | 0.42 | Partially mixed |

### Weighted Gini after a split
$$Gini_{split} = \frac{n_{left}}{n} \cdot Gini_{left} + \frac{n_{right}}{n} \cdot Gini_{right}$$

The tree picks the split that **minimises** this weighted Gini.

In [ ]:
# === Concept 2: Gini Impurity — full derivation from scratch ===

def gini(y):
    """Gini impurity for array of labels. Formula: 1 - sum(p_i^2)"""
    if len(y) == 0: return 0.0
    _, counts = np.unique(y, return_counts=True)
    ps = counts / len(y)
    return 1 - np.sum(ps**2)

def weighted_gini_split(y_left, y_right):
    """Gini after splitting parent into left and right children."""
    n = len(y_left) + len(y_right)
    return (len(y_left)/n) * gini(y_left) + (len(y_right)/n) * gini(y_right)

# Demonstrate with a concrete example
examples = [
    ("All class 0",        np.array([0,0,0,0,0])),
    ("80% class 0",        np.array([0,0,0,0,1])),
    ("60/40 split",        np.array([0,0,0,1,1])),
    ("50/50 (worst)",      np.array([0,0,0,1,1,1])),
    ("Mixed 3-class",      np.array([0,1,2,0,1,2])),
]

print("=== Gini Impurity Values ===")
for name, y_ex in examples:
    g = gini(y_ex)
    counts = dict(zip(*np.unique(y_ex, return_counts=True)))
    print(f"  {name:<22} labels={y_ex}  Gini={g:.4f}")

# Visualise Gini vs proportion
p = np.linspace(0.001, 0.999, 500)
gini_vals = 2*p*(1-p)   # for binary: 1-(p²+(1-p)²) = 2p(1-p)

plt.figure(figsize=(9, 4))
plt.plot(p, gini_vals, color='steelblue', linewidth=2.5)
plt.fill_between(p, gini_vals, alpha=0.15, color='steelblue')
plt.axvline(0.5, color='red', linestyle='--', label='50/50 — worst split (Gini=0.5)')
plt.axhline(0,   color='green', linestyle='--', label='Pure node — best (Gini=0.0)')
plt.xlabel('Proportion of Class 1 in node')
plt.ylabel('Gini Impurity')
plt.title('Gini Impurity vs Class Proportion (binary case)')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

# Show split comparison
print()
print("=== Comparing Two Splits ===")
y_parent = np.array([0,0,0,0,1,1,1,1])   # 50/50 parent
print(f"Parent Gini: {gini(y_parent):.4f}")

# Good split
yl_good = np.array([0,0,0,0]); yr_good = np.array([1,1,1,1])
print(f"\nGood split: left={yl_good} | right={yr_good}")
print(f"  Weighted Gini = {weighted_gini_split(yl_good, yr_good):.4f} (0 = perfect)")

# Bad split
yl_bad = np.array([0,0,1,1]); yr_bad = np.array([0,0,1,1])
print(f"\nBad split:  left={yl_bad} | right={yr_bad}")
print(f"  Weighted Gini = {weighted_gini_split(yl_bad, yr_bad):.4f} (same as parent)")

---
## Concept 3 — Entropy & Information Gain

Entropy is an alternative to Gini. From information theory — measures surprise/disorder.

$$Entropy = -\sum_{i=1}^{C} p_i \log_2(p_i)$$

**Information Gain** = how much entropy is reduced by a split:
$$IG = Entropy(parent) - \frac{n_{left}}{n} \cdot Entropy(left) - \frac{n_{right}}{n} \cdot Entropy(right)$$

Higher IG = better split. The tree maximises IG.

### Gini vs Entropy
| | Gini | Entropy |
|---|---|---|
| Speed | Faster (no log) | Slower |
| Behaviour | Slightly favours larger partitions | More balanced |
| Result | Usually identical | Usually identical |
| sklearn param | `criterion='gini'` | `criterion='entropy'` |

In [ ]:
# === Concept 3: Entropy and Information Gain from scratch ===

def entropy(y):
    """Shannon entropy. Formula: -sum(p_i * log2(p_i))"""
    if len(y) == 0: return 0.0
    _, counts = np.unique(y, return_counts=True)
    ps = counts / len(y)
    ps = ps[ps > 0]   # avoid log(0)
    return -np.sum(ps * np.log2(ps))

def information_gain(y_parent, y_left, y_right):
    """How much entropy drops after splitting parent into left and right."""
    n = len(y_parent)
    weighted_child_entropy = (
        (len(y_left) /n) * entropy(y_left) +
        (len(y_right)/n) * entropy(y_right)
    )
    return entropy(y_parent) - weighted_child_entropy

# Entropy values
examples = [
    ("All class 0",   np.array([0,0,0,0,0])),
    ("80% class 0",   np.array([0,0,0,0,1])),
    ("50/50",         np.array([0,0,0,1,1,1])),
    ("3-class equal", np.array([0,1,2,0,1,2])),
]

print("=== Entropy Values ===")
for name, y_ex in examples:
    print(f"  {name:<20}: Entropy={entropy(y_ex):.4f}  Gini={gini(y_ex):.4f}")

# Compare Gini vs Entropy curve
p = np.linspace(0.001, 0.999, 500)
gini_curve    = 2*p*(1-p)
entropy_curve = -(p*np.log2(p) + (1-p)*np.log2(1-p))

plt.figure(figsize=(9, 4))
plt.plot(p, gini_curve,         color='steelblue', linewidth=2.5, label='Gini (×2 for scale)')
plt.plot(p, entropy_curve/2,    color='coral',     linewidth=2.5, linestyle='--',
         label='Entropy / 2 (rescaled)')
plt.xlabel('Proportion of class 1'); plt.ylabel('Impurity')
plt.title('Gini vs Entropy — Almost Identical Shape\n'
          '(That\'s why criterion choice rarely matters in practice)')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

# Information Gain example
y_p = np.array([0,0,0,0,1,1,1,1])
yl  = np.array([0,0,0,0])
yr  = np.array([1,1,1,1])
print(f"\n=== Information Gain Example ===")
print(f"Parent entropy:  {entropy(y_p):.4f}")
print(f"Left entropy:    {entropy(yl):.4f}")
print(f"Right entropy:   {entropy(yr):.4f}")
print(f"Information Gain: {information_gain(y_p, yl, yr):.4f}  (1.0 = perfect split)")

---
## Concept 4 — How the Tree Finds the Best Split (Exhaustive Search)

For every node, the algorithm does this:

```
For each feature f in all features:
    Sort training points by feature f
    For each unique value v of feature f:
        Left  = points where f <= v
        Right = points where f > v
        Compute weighted Gini (or IG)
    
    Record the best (v, Gini) for feature f

Pick the feature f and threshold v with lowest Gini overall
Split node there
```

This is O(n × d × log n) per node — brute force exhaustive search.

In [ ]:
# === Concept 4: Finding the best split from scratch ===

def find_best_split(X, y, criterion='gini'):
    """
    Find the best feature and threshold to split on.
    Returns: (best_feature_index, best_threshold, best_score)
    """
    n_features = X.shape[1]
    best_score  = float('inf')   # we minimise Gini
    best_feat   = None
    best_thresh = None

    for feat in range(n_features):
        # Get unique sorted values for this feature
        thresholds = np.unique(X[:, feat])

        for thresh in thresholds:
            # Split
            left_mask  = X[:, feat] <= thresh
            right_mask = ~left_mask

            y_left  = y[left_mask]
            y_right = y[right_mask]

            # Skip trivial splits
            if len(y_left) == 0 or len(y_right) == 0:
                continue

            score = weighted_gini_split(y_left, y_right)

            if score < best_score:
                best_score  = score
                best_feat   = feat
                best_thresh = thresh

    return best_feat, best_thresh, best_score


# Demo with real data
np.random.seed(42)
X_ex = np.array([[2.5, 1.0], [3.0, 2.0], [1.5, 3.0],
                  [5.0, 1.5], [6.0, 2.5], [4.5, 3.5]], dtype=float)
y_ex = np.array([0, 0, 0, 1, 1, 1])

feat, thresh, score = find_best_split(X_ex, y_ex)
print("=== Best Split Search ===")
print(f"Best feature:    {feat} ({'Feature 0' if feat==0 else 'Feature 1'})")
print(f"Best threshold:  {thresh}")
print(f"Gini after split: {score:.4f}")
print()

# Show all possible splits and their Gini scores
print("All possible splits explored:")
for f in range(2):
    for t in np.unique(X_ex[:,f]):
        yl = y_ex[X_ex[:,f] <= t]
        yr = y_ex[X_ex[:,f] > t]
        if len(yl)>0 and len(yr)>0:
            g = weighted_gini_split(yl, yr)
            marker = " ← BEST" if f==feat and t==thresh else ""
            print(f"  Feature {f} <= {t:.1f}: Gini={g:.4f}{marker}")

---
## Concept 5 — Overfitting: The Unlimited Tree Problem

A fully grown Decision Tree with no depth limit will keep splitting until every leaf has exactly 1 sample — or until all samples in a node belong to the same class.

**Result:** 100% training accuracy, terrible test accuracy.

**Why?** It memorises noise in the training data rather than learning the true pattern.

### What a fully grown tree looks like
- Many tiny leaves, each with 1-2 samples
- Extremely jagged decision boundary
- Zero generalisation

In [ ]:
# === Concept 5: Overfitting vs Pruned — side by side ===

X_of, y_of = make_moons(n_samples=300, noise=0.3, random_state=42)
X_of = StandardScaler().fit_transform(X_of)
X_oftr, X_ofte, y_oftr, y_ofte = train_test_split(X_of, y_of, test_size=0.3,
                                                    random_state=42, stratify=y_of)

configs = [
    (None,  'Unlimited depth\n(overfits — memorises noise)'),
    (2,     'max_depth=2\n(underfits — too simple)'),
    (5,     'max_depth=5\n(good balance)'),
]

fig, axes = plt.subplots(2, 3, figsize=(17, 10))
xx, yy = np.meshgrid(
    np.linspace(X_of[:,0].min()-0.3, X_of[:,0].max()+0.3, 250),
    np.linspace(X_of[:,1].min()-0.3, X_of[:,1].max()+0.3, 250)
)

for col, (depth, title) in enumerate(configs):
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_oftr, y_oftr)
    Z = dt.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    # Decision boundary
    axes[0][col].contourf(xx, yy, Z, alpha=0.25, cmap='RdBu')
    axes[0][col].scatter(X_oftr[y_oftr==0,0], X_oftr[y_oftr==0,1], c='steelblue', s=20, alpha=0.7)
    axes[0][col].scatter(X_oftr[y_oftr==1,0], X_oftr[y_oftr==1,1], c='coral',     s=20, alpha=0.7)
    axes[0][col].set_title(
        f'{title}\nDepth={dt.get_depth()} | Leaves={dt.get_n_leaves()}\n'
        f'Train={dt.score(X_oftr,y_oftr)*100:.1f}% | Test={dt.score(X_ofte,y_ofte)*100:.1f}%',
        fontsize=9)
    axes[0][col].grid(True, alpha=0.2)

    # Tree structure
    plot_tree(dt, max_depth=3, filled=True, rounded=True, fontsize=6, ax=axes[1][col])
    axes[1][col].set_title(f'Tree Structure (depth={dt.get_depth()})', fontsize=9)

plt.suptitle('Overfitting, Underfitting, and Good Fit in Decision Trees', fontsize=13)
plt.tight_layout(); plt.show()

---
## Concept 6 — Pruning Parameters

Pruning = restricting how much the tree grows.

| Parameter | What it controls | Effect of increasing |
|---|---|---|
| `max_depth` | Maximum levels in tree | Simpler tree, more bias |
| `min_samples_split` | Min samples needed to split a node | Fewer, simpler splits |
| `min_samples_leaf` | Min samples required in each leaf | Smoother boundaries |
| `max_leaf_nodes` | Hard cap on number of leaves | Direct size control |
| `min_impurity_decrease` | Only split if Gini improves by at least this much | Fewer splits |

### Rule of thumb
Start with `max_depth` in range 3-10. Use `GridSearchCV` to tune.

In [ ]:
# === Concept 6: Pruning — effect of each parameter ===

# Generate data
X_pr, y_pr = make_moons(n_samples=500, noise=0.25, random_state=42)
X_pr = StandardScaler().fit_transform(X_pr)
X_prtr, X_prte, y_prtr, y_prte = train_test_split(X_pr, y_pr, test_size=0.25,
                                                    random_state=42, stratify=y_pr)

# max_depth sweep
depths = list(range(1, 20))
train_accs, test_accs = [], []
for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_prtr, y_prtr)
    train_accs.append(dt.score(X_prtr, y_prtr))
    test_accs.append(dt.score(X_prte, y_prte))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(depths, train_accs, color='steelblue', marker='o', ms=5, label='Train')
axes[0].plot(depths, test_accs,  color='coral',     marker='o', ms=5, label='Test')
axes[0].axvline(depths[np.argmax(test_accs)], color='black', linestyle=':',
                label=f'Best test depth={depths[np.argmax(test_accs)]}')
axes[0].set_title('max_depth Effect'); axes[0].set_xlabel('max_depth')
axes[0].set_ylabel('Accuracy'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

# min_samples_leaf sweep
leaf_sizes = [1, 2, 5, 10, 20, 30, 50]
tr_l, te_l = [], []
for ms in leaf_sizes:
    dt = DecisionTreeClassifier(min_samples_leaf=ms, random_state=42)
    dt.fit(X_prtr, y_prtr)
    tr_l.append(dt.score(X_prtr, y_prtr))
    te_l.append(dt.score(X_prte, y_prte))

axes[1].plot(leaf_sizes, tr_l, color='steelblue', marker='o', ms=5, label='Train')
axes[1].plot(leaf_sizes, te_l, color='coral',     marker='o', ms=5, label='Test')
axes[1].set_title('min_samples_leaf Effect'); axes[1].set_xlabel('min_samples_leaf')
axes[1].set_ylabel('Accuracy'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Pruning Parameters — Controlling Tree Complexity', fontsize=12)
plt.tight_layout(); plt.show()

---
## Concept 7 — Feature Importance

Feature importance = how much each feature reduced Gini impurity across all splits in the tree.

$$Importance(f) = \sum_{nodes\ using\ f} \frac{n_{node}}{n_{total}} \cdot (Gini_{before} - Gini_{after})$$

Normalised so all importances sum to 1.

**What it tells you:** the higher the importance, the more the tree relied on that feature to reduce error. It does NOT tell you the direction of effect.

In [ ]:
# === Concept 7: Feature importance — derivation and visualisation ===

from sklearn.datasets import load_iris
iris = load_iris()
X_ir, y_ir = iris.data, iris.target

dt_ir = DecisionTreeClassifier(max_depth=4, random_state=42)
dt_ir.fit(X_ir, y_ir)

importances = dt_ir.feature_importances_
feat_names  = iris.feature_names

# Manual calculation explanation
print("=== Feature Importances ===")
for name, imp in sorted(zip(feat_names, importances), key=lambda x: -x[1]):
    bar = '█' * int(imp * 50)
    print(f"  {name:<30} {bar}  {imp*100:.2f}%")

print()
print("Interpretation:")
print("  Higher % = tree used this feature more to reduce impurity")
print("  Does NOT tell you if high value = more positive or more negative")

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = plt.cm.RdYlGn([imp for imp in sorted(importances, reverse=True)])
sorted_feats = [f for _, f in sorted(zip(importances, feat_names), reverse=True)]
sorted_imps  = sorted(importances, reverse=True)

axes[0].barh(sorted_feats, sorted_imps, color=colors)
axes[0].set_xlabel('Importance Score'); axes[0].set_title('Feature Importances (Iris)')
axes[0].invert_yaxis(); axes[0].grid(True, alpha=0.3, axis='x')

# Show the tree
plot_tree(dt_ir, feature_names=feat_names, class_names=iris.target_names,
          filled=True, rounded=True, fontsize=8, ax=axes[1], max_depth=3)
axes[1].set_title('Tree Structure — most important features appear near the root')

plt.tight_layout(); plt.show()

print()
print("Note: features with higher importance appear closer to the root (top) of the tree.")
print("The root split = the single most powerful separator in the entire dataset.")

---
## Concept 8 — Decision Tree for Regression (CART)

Decision Trees work for regression too. Instead of predicting a class label at each leaf, the tree predicts the **mean of all training samples in that leaf.**

$$\hat{y}_{leaf} = \frac{1}{|leaf|} \sum_{i \in leaf} y_i$$

For regression splits, instead of Gini/Entropy, the criterion is **Mean Squared Error (MSE):**
$$MSE_{split} = \frac{n_{left}}{n} MSE_{left} + \frac{n_{right}}{n} MSE_{right}$$

In [ ]:
# === Concept 8: Decision Tree Regression — piecewise constant predictions ===

np.random.seed(42)
X_rg = np.linspace(0, 8, 200).reshape(-1, 1)
y_rg = np.sin(X_rg).ravel() + 0.3*np.random.randn(200)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['coral', 'steelblue', 'mediumseagreen']

for ax, depth, color in zip(axes, [1, 3, 10], colors):
    dtr = DecisionTreeRegressor(max_depth=depth, random_state=42)
    dtr.fit(X_rg, y_rg)
    y_hat = dtr.predict(X_rg)

    ax.scatter(X_rg, y_rg, c='black', s=8, alpha=0.4, label='Training data')
    ax.plot(X_rg, y_hat, color=color, linewidth=2.5, label=f'DT Regression depth={depth}')
    ax.set_title(f'max_depth={depth}\n'
                 f'Leaves={dtr.get_n_leaves()} | '
                 f'Each leaf predicts the mean of its samples', fontsize=9)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle('Decision Tree Regression — Piecewise Constant Predictions\n'
             'Shallow=underfit (flat steps), Deep=overfit (memorises each point)', fontsize=11)
plt.tight_layout(); plt.show()

print("Key insight: DT regression creates a step-function, not a smooth curve.")
print("Each step = one leaf. More leaves = more steps = closer fit to data.")
print("Contrast with linear regression which always gives a single straight line.")

---
## Concept 9 — Instability

Decision Trees have **high variance** — small changes in training data can produce a completely different tree.

**Why?** Each split is chosen greedily. If one split changes, all splits below it change too.

This is the core reason Random Forests were invented — average many unstable trees → stable result.

In [ ]:
# === Concept 9: Instability — show how tree changes with tiny data change ===

np.random.seed(42)
X_st, y_st = make_moons(n_samples=150, noise=0.25, random_state=42)
X_st = StandardScaler().fit_transform(X_st)

fig, axes = plt.subplots(2, 3, figsize=(17, 10))
xx, yy = np.meshgrid(
    np.linspace(X_st[:,0].min()-0.3, X_st[:,0].max()+0.3, 200),
    np.linspace(X_st[:,1].min()-0.3, X_st[:,1].max()+0.3, 200)
)

# Train 6 trees — each with a different random 10-sample subset removed
for col in range(3):
    # Remove 10 random points — simulates slightly different training data
    drop_idx = np.random.choice(len(X_st), 10, replace=False)
    mask = np.ones(len(X_st), dtype=bool); mask[drop_idx] = False
    X_sub = X_st[mask]; y_sub = y_st[mask]

    dt_sub = DecisionTreeClassifier(max_depth=5, random_state=42)
    dt_sub.fit(X_sub, y_sub)
    Z = dt_sub.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    axes[0][col].contourf(xx, yy, Z, alpha=0.25, cmap='RdBu')
    axes[0][col].scatter(X_sub[y_sub==0,0], X_sub[y_sub==0,1], c='steelblue', s=20, alpha=0.7)
    axes[0][col].scatter(X_sub[y_sub==1,0], X_sub[y_sub==1,1], c='coral',     s=20, alpha=0.7)
    axes[0][col].set_title(f'Version {col+1}: 10 points removed\nBoundary changes significantly', fontsize=9)
    axes[0][col].grid(True, alpha=0.2)

    plot_tree(dt_sub, max_depth=3, filled=True, rounded=True, fontsize=6, ax=axes[1][col])
    axes[1][col].set_title(f'Tree structure — Version {col+1}', fontsize=9)

plt.suptitle('Decision Tree Instability\n'
             'Removing just 10 points produces completely different trees', fontsize=12)
plt.tight_layout(); plt.show()
print("This instability is EXACTLY why Random Forest was invented.")
print("500 unstable trees averaged together → stable prediction.")

---
## Concept 10 — Full Decision Tree from Scratch

A minimal but complete CART implementation.

In [ ]:
# === Concept 10: Full Decision Tree from scratch ===

class TreeNode:
    """One node in the Decision Tree."""
    def __init__(self):
        self.feature    = None    # which feature we split on
        self.threshold  = None    # the split value
        self.left       = None    # left child (feature <= threshold)
        self.right      = None    # right child (feature > threshold)
        self.is_leaf    = False   # True if this node gives a final prediction
        self.prediction = None    # class label at leaf


class DecisionTreeScratch:
    """CART Decision Tree Classifier from scratch."""

    def __init__(self, max_depth=5, min_samples_leaf=1):
        self.max_depth        = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.root             = None

    def _gini(self, y):
        if len(y) == 0: return 0.0
        _, counts = np.unique(y, return_counts=True)
        ps = counts / len(y)
        return 1 - np.sum(ps**2)

    def _best_split(self, X, y):
        best_gini  = float('inf')
        best_feat  = None
        best_thresh = None

        for feat in range(X.shape[1]):
            thresholds = np.unique(X[:, feat])
            for thresh in thresholds:
                left  = y[X[:,feat] <= thresh]
                right = y[X[:,feat] >  thresh]
                if len(left) < self.min_samples_leaf or len(right) < self.min_samples_leaf:
                    continue
                n = len(y)
                g = (len(left)/n)*self._gini(left) + (len(right)/n)*self._gini(right)
                if g < best_gini:
                    best_gini   = g
                    best_feat   = feat
                    best_thresh = thresh

        return best_feat, best_thresh

    def _build(self, X, y, depth):
        node = TreeNode()

        # Stopping conditions → make a leaf
        if (depth >= self.max_depth or
            len(np.unique(y)) == 1 or
            len(y) <= self.min_samples_leaf):
            node.is_leaf    = True
            node.prediction = np.bincount(y).argmax()
            return node

        feat, thresh = self._best_split(X, y)

        # If no valid split found → leaf
        if feat is None:
            node.is_leaf    = True
            node.prediction = np.bincount(y).argmax()
            return node

        node.feature   = feat
        node.threshold = thresh

        left_mask  = X[:,feat] <= thresh
        right_mask = ~left_mask

        node.left  = self._build(X[left_mask],  y[left_mask],  depth+1)
        node.right = self._build(X[right_mask], y[right_mask], depth+1)
        return node

    def fit(self, X, y):
        X, y = np.array(X, float), np.array(y, int)
        self.root = self._build(X, y, depth=0)
        return self

    def _predict_one(self, x, node):
        if node.is_leaf:
            return node.prediction
        if x[node.feature] <= node.threshold:
            return self._predict_one(x, node.left)
        else:
            return self._predict_one(x, node.right)

    def predict(self, X):
        return np.array([self._predict_one(x, self.root) for x in np.array(X, float)])

    def score(self, X, y):
        return (self.predict(X) == np.array(y)).mean()


# ── Test vs sklearn ──
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer()
X_bc, y_bc = data.data[:, :6], data.target
X_bctr, X_bcte, y_bctr, y_bcte = train_test_split(X_bc, y_bc, test_size=0.2, random_state=42)

our_dt = DecisionTreeScratch(max_depth=5, min_samples_leaf=2)
our_dt.fit(X_bctr, y_bctr)

sk_dt = DecisionTreeClassifier(max_depth=5, min_samples_leaf=2, random_state=42)
sk_dt.fit(X_bctr, y_bctr)

print("=== From Scratch vs sklearn ===")
print(f"Our DT accuracy:     {our_dt.score(X_bcte, y_bcte)*100:.2f}%")
print(f"sklearn DT accuracy: {sk_dt.score(X_bcte, y_bcte)*100:.2f}%")
print("(Small difference is OK — sklearn uses slightly different tie-breaking)")

---
## Summary — Every Decision Tree Concept at a Glance

| Concept | Key point |
|---|---|
| Core idea | Flowchart of yes/no questions. Each split maximises class purity. |
| Gini impurity | 1 - Σpᵢ². 0=pure, 0.5=worst. Tree minimises weighted Gini. |
| Entropy | -Σpᵢlog₂(pᵢ). Information Gain = entropy drop after split. |
| Best split search | Exhaustive: try every feature, every threshold. O(n × d × log n). |
| CART | Binary splits only. Greedy. No look-ahead. |
| No scaling needed | Splits use thresholds (>, <) not distances. Scale has no effect. |
| Overfitting | Unlimited tree = 100% train accuracy, terrible test accuracy. |
| max_depth | Main pruning parameter. Tune with cross-validated F1. |
| min_samples_leaf | Min samples per leaf. Larger = simpler tree. |
| Feature importance | Gini reduction weighted by node size. Sums to 1. |
| Regression | Predict mean of leaf samples instead of class. MSE for splits. |
| Instability | Small data change → completely different tree. |
| Instability fix | → Random Forest (average 100s of trees). |

### When to use Decision Trees
- When you need human-readable rules (HR, medical, legal)
- Mixed feature types (numeric + categorical together)
- Quick interpretable baseline
- Feature importance needed

### When NOT to use
- When accuracy is the priority → use Random Forest
- When boundary is very complex and smooth
- Small datasets (instability is worse with less data)